# Tree-Based Models Prototype

In [1]:
import pandas as pd
import datasets
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder
import warnings
import json
import os
import io
from datasets import load_dataset

In [2]:
ds = load_dataset("thiru1711/Financial_Transactions")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
for feature_name, feature_type in ds['train'].features.items():
    print(f"Column: {feature_name}, Data Type: {feature_type.dtype if hasattr(feature_type, 'dtype') else str(feature_type)}")

Column: transaction_id, Data Type: string
Column: date, Data Type: timestamp[ns]
Column: card_id, Data Type: string
Column: amount, Data Type: float32
Column: use_chip, Data Type: string
Column: merchant_id, Data Type: int64
Column: merchant_city, Data Type: string
Column: merchant_state, Data Type: string
Column: zip, Data Type: float64
Column: mcc, Data Type: string
Column: errors, Data Type: string
Column: is_fraud, Data Type: int64
Column: card_brand, Data Type: string
Column: card_type, Data Type: string
Column: card_number, Data Type: int64
Column: expires, Data Type: string
Column: cvv, Data Type: int16
Column: has_chip, Data Type: string
Column: num_cards_issued, Data Type: int64
Column: credit_limit, Data Type: float32
Column: acct_open_date, Data Type: string
Column: year_pin_last_changed, Data Type: int64
Column: card_on_dark_web, Data Type: string
Column: current_age, Data Type: int64
Column: retirement_age, Data Type: int64
Column: birth_year, Data Type: int64
Column: birt

In [4]:
df_raw = ds['train'].to_pandas()

In [5]:
cond = (
    (df_raw['is_fraud'] == 1) |
    (df_raw['amount'] < 0) |
    (df_raw['errors'].notnull()) |
    (df_raw['card_brand'] == 'Amex')
)
df_raw = df_raw[~cond].reset_index(drop=True)

In [6]:
df_raw

,transaction_id,date,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,mcc_description
0,7475328,2010-01-01 00:02:00,4575,14.570000,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,...,Male,604 Pine Street,40.80,-91.12,18076.0,36853.0,112139.0,834,5,Department Stores
1,7475329,2010-01-01 00:02:00,102,80.000000,Swipe Transaction,27092,Vista,CA,92084.0,4829,...,Male,2379 Forest Lane,33.18,-117.29,16894.0,34449.0,36540.0,686,3,Money Transfer
2,7475331,2010-01-01 00:05:00,2860,200.000000,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,...,Female,903 Hill Boulevard,41.42,-87.35,26168.0,53350.0,128676.0,685,5,Money Transfer
3,7475332,2010-01-01 00:06:00,3915,46.410000,Swipe Transaction,13051,Harwood,MD,20776.0,5813,...,Male,166 River Drive,38.86,-76.60,33529.0,68362.0,96182.0,711,2,Drinking Places (Alcoholic Beverages)
4,7475333,2010-01-01 00:07:00,165,4.810000,Swipe Transaction,20519,Bronx,NY,10464.0,5942,...,Female,14780 Plum Lane,40.84,-73.87,25537.0,52065.0,98613.0,828,5,Book Stores
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11638657,23761868,2019-10-31 23:56:00,2379,1.110000,Chip Transaction,86438,West Covina,CA,91792.0,5499,...,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5,Miscellaneous Food Stores
11638658,23761869,2019-10-31 23:56:00,2066,12.800000,Online Transaction,39261,ONLINE,None,NaN,5815,...,Male,6076 Bayview Boulevard,43.06,-87.96,9995.0,20377.0,12092.0,789,4,"Digital Goods - Media, Books, Apps"
11638659,23761870,2019-10-31 23:57:00,1031,40.439999,Swipe Transaction,2925,Allen,TX,75002.0,4900,...,Female,7927 Plum Lane,33.10,-96.66,32580.0,78329.0,40161.0,720,3,"Utilities - Electric, Gas, Water, Sanitary"
11638660,23761873,2019-10-31 23:58:00,5443,4.000000,Chip Transaction,46284,Daly City,CA,94014.0,5411,...,Female,5887 Seventh Lane,37.68,-122.43,23752.0,48430.0,62384.0,716,2,"Grocery Stores, Supermarkets"


In [7]:
# Drop columns not needed
drop_cols = [
    # PII/Security Fields
    'card_number', 'cvv', 'expires', 'address',

    # Cardholder Demographics
    'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender',
    'latitude', 'longitude', 'per_capita_income', 'yearly_income',
    'total_debt', 'credit_score', 'num_credit_cards','credit_limit',

    # Account Metadata
    'card_id', 'acct_open_date', 'year_pin_last_changed',
    'card_on_dark_web', 'num_cards_issued',

    # Geographical features
    'merchant_state', 'zip', 'merchant_city', 'has_chip', 'use_chip', 'is_fraud', 'errors'
]

df_raw = df_raw.drop(columns=drop_cols)

In [8]:
df_raw

,transaction_id,date,amount,merchant_id,mcc,card_brand,card_type,mcc_description
0,7475328,2010-01-01 00:02:00,14.570000,67570,5311,Mastercard,Credit,Department Stores
1,7475329,2010-01-01 00:02:00,80.000000,27092,4829,Mastercard,Debit,Money Transfer
2,7475331,2010-01-01 00:05:00,200.000000,27092,4829,Mastercard,Debit,Money Transfer
3,7475332,2010-01-01 00:06:00,46.410000,13051,5813,Visa,Debit,Drinking Places (Alcoholic Beverages)
4,7475333,2010-01-01 00:07:00,4.810000,20519,5942,Mastercard,Debit (Prepaid),Book Stores
...,...,...,...,...,...,...,...,...
11638657,23761868,2019-10-31 23:56:00,1.110000,86438,5499,Mastercard,Debit,Miscellaneous Food Stores
11638658,23761869,2019-10-31 23:56:00,12.800000,39261,5815,Mastercard,Debit,"Digital Goods - Media, Books, Apps"
11638659,23761870,2019-10-31 23:57:00,40.439999,2925,4900,Mastercard,Debit,"Utilities - Electric, Gas, Water, Sanitary"
11638660,23761873,2019-10-31 23:58:00,4.000000,46284,5411,Visa,Debit,"Grocery Stores, Supermarkets"


In [9]:
# Convert 'mcc' column to numeric, coercing errors to NaN
df_raw['mcc'] = pd.to_numeric(df_raw['mcc'], errors='coerce')

# Filter out rows where 'mcc' is NaN after conversion (if any)
df_raw = df_raw.dropna(subset=['mcc'])

In [10]:
# Rename "Debit (Prepaid)" to "Prepaid" in card_type column
df_raw['card_type'] = df_raw['card_type'].replace('Debit (Prepaid)', 'Prepaid')

In [11]:
cost_type_df = pd.read_csv('/content/cost_type_id.csv')

# Ensure mcc is same type in both dataframes
df_raw['mcc'] = df_raw['mcc'].astype(float)
cost_type_df['mcc'] = pd.to_numeric(cost_type_df['mcc'], errors='coerce')

# Filter to only card brands that exist in cost_type_df
df_raw = df_raw[df_raw['card_brand'].isin(['Visa', 'Mastercard'])].copy()

# Separate rules
cost_general = cost_type_df[cost_type_df['mcc'].isna()].copy()
cost_specific = cost_type_df[cost_type_df['mcc'].notna()].copy()

# Process small (<=5): general rules
df_small = df_raw[df_raw['amount'] <= 5].merge(
    cost_general,
    left_on=['card_brand', 'card_type'],
    right_on=['card_brand', 'card_type'],
    how='left',
    suffixes=('', '_cost')
)
df_small = df_small[
    (df_small['amount'] >= df_small['min_transaction_amt']) &
    (df_small['amount'] <= df_small['max_transaction_amt']) &
    (df_small['cost_type_ID'].notna())
]
df_small = df_small.sort_values('transaction_id').drop_duplicates('transaction_id', keep='first')

# Process large (>5): mcc-specific rules
df_large = df_raw[df_raw['amount'] > 5].merge(
    cost_specific,
    left_on=['card_brand', 'card_type', 'mcc'],
    right_on=['card_brand', 'card_type', 'mcc'],
    how='left',
    suffixes=('', '_cost')
)
df_large = df_large[
    (df_large['amount'] >= df_large['min_transaction_amt']) &
    (df_large['amount'] <= df_large['max_transaction_amt']) &
    (df_large['cost_type_ID'].notna())
]
df_large = df_large.sort_values('transaction_id').drop_duplicates('transaction_id', keep='first')

# Combine results and map back to original df
result_map = pd.concat([
    df_small[['transaction_id', 'cost_type_ID']],
    df_large[['transaction_id', 'cost_type_ID']]
]).set_index('transaction_id')['cost_type_ID']

# Drop old cost_type_ID if exists
if 'cost_type_ID' in df_raw.columns:
    df_raw = df_raw.drop('cost_type_ID', axis=1)

df_raw['cost_type_ID'] = df_raw['transaction_id'].map(result_map)

print(f"Total transactions: {len(df_raw)}")
print(f"Rows with cost_type_ID: {df_raw['cost_type_ID'].notna().sum()}")
print(f"Rows without match: {df_raw['cost_type_ID'].isna().sum()}")
print(f"\nBreakdown of unmatched:")
print(f"  - Negative amounts: {(df_raw['amount'] < 0).sum()}")
print(f"  - Amex cards: {(df_raw['card_brand'] == 'Amex').sum()}")
print(f"  - Discover cards: {(df_raw['card_brand'] == 'Discover').sum()}")
print(f"  - Other: {df_raw['cost_type_ID'].isna().sum() - (df_raw['amount'] < 0).sum() - (df_raw['card_brand'] == 'Amex').sum() - (df_raw['card_brand'] == 'Discover').sum()}")


Total transactions: 11322588
Rows with cost_type_ID: 4430446
Rows without match: 6892142

Breakdown of unmatched:
  - Negative amounts: 0
  - Amex cards: 0
  - Discover cards: 0
  - Other: 6892142


In [ ]:
# Prepare lookup arrays indexed by cost_type_ID
cost_type_df['subtotal_fee_percent_clean'] = cost_type_df['subtotal_fee_percent'].str.rstrip('%').astype(float) / 100
cost_type_df['subtotal_fee_dollars_clean'] = cost_type_df['subtotal_fee_dollars'].str.lstrip('$').astype(float)

# Create dictionaries for O(1) lookup
fee_dollars_map = dict(zip(cost_type_df['cost_type_ID'], cost_type_df['subtotal_fee_dollars_clean']))
fee_percent_map = dict(zip(cost_type_df['cost_type_ID'], cost_type_df['subtotal_fee_percent_clean']))

# Vectorized calculation using map (no merge, no temporary columns)
df_raw['proc_cost'] = df_raw['cost_type_ID'].map(fee_dollars_map) + (df_raw['cost_type_ID'].map(fee_percent_map) * df_raw['amount'])

print(f"✓ proc_cost calculated for {df_raw['proc_cost'].notna().sum():,} transactions")


: 

In [ ]:
conn = sqlite3.connect("denzel.db")

# Load transactions and extract year_month
transactions_df = pd.read_sql("SELECT merchant_id, amount, proc_cost, date FROM transactions", conn)

transactions_df["year_month"] = pd.to_datetime(transactions_df["date"]).dt.strftime("%Y-%m")

# Group and aggregate
merchants_df = (
    transactions_df
    .groupby(["merchant_id", "year_month"], as_index=False)
    .agg(
        TPV=("amount",    "sum"),
        TPC=("proc_cost", "sum")
    )
)

# Calculate TCR as a percentage
merchants_df["TCR"] = (merchants_df["TPC"] / merchants_df["TPV"] * 100).round(4)

# Write to denzel.db
merchants_df.to_sql("merchants", conn, if_exists="replace", index=False)

conn.close()
print(f"Done! {len(merchants_df)} rows written to merchants table.")

In [ ]:
conn = sqlite3.connect("denzel.db")

# Load transactions — now including cost_type_ID
transactions_df = pd.read_sql(
    "SELECT merchant_id, amount, proc_cost, date, cost_type_ID FROM transactions", conn
)
transactions_df["year_month"] = pd.to_datetime(transactions_df["date"]).dt.strftime("%Y-%m")

# ── Base aggregation ──────────────────────────────────────────────────────────
merchants_df = (
    transactions_df
    .groupby(["merchant_id", "year_month"], as_index=False)
    .agg(
        TPV        = ("amount",    "sum"),
        TPC        = ("proc_cost", "sum"),
        total_txns = ("amount",    "count")   # used for pct calculations, dropped later
    )
)
merchants_df["TCR"] = (merchants_df["TPC"] / merchants_df["TPV"] * 100).round(4)

# ── pct_small_ticket ──────────────────────────────────────────────────────────
small_ticket_ids = [1, 2, 3, 4, 5, 6, 24, 25, 26, 27]

small_df = (
    transactions_df[transactions_df["cost_type_ID"].isin(small_ticket_ids)]
    .groupby(["merchant_id", "year_month"])
    .size()
    .reset_index(name="small_ticket_count")
)
merchants_df = merchants_df.merge(small_df, on=["merchant_id", "year_month"], how="left")
merchants_df["small_ticket_count"] = merchants_df["small_ticket_count"].fillna(0)
merchants_df["pct_small_ticket"] = (
    merchants_df["small_ticket_count"] / merchants_df["total_txns"] * 100
).round(4)
merchants_df.drop(columns=["small_ticket_count"], inplace=True)

# ── pct_cost_type_[id] ────────────────────────────────────────────────────────
cost_counts = (
    transactions_df
    .groupby(["merchant_id", "year_month", "cost_type_ID"])
    .size()
    .reset_index(name="ct_count")
)
cost_counts = cost_counts.merge(
    merchants_df[["merchant_id", "year_month", "total_txns"]],
    on=["merchant_id", "year_month"]
)
cost_counts["pct"] = (cost_counts["ct_count"] / cost_counts["total_txns"] * 100).round(4)

cost_pivot = cost_counts.pivot_table(
    index=["merchant_id", "year_month"],
    columns="cost_type_ID",
    values="pct",
    fill_value=0
).reset_index()

# Rename columns: cost_type_ID int → pct_cost_type_X
cost_pivot.columns = [
    f"pct_cost_type_{int(c)}" if isinstance(c, (int, float)) else c
    for c in cost_pivot.columns
]

merchants_df = merchants_df.merge(cost_pivot, on=["merchant_id", "year_month"], how="left")

# ── std_TCR (std of all OTHER merchants' TCR in same yyyy-mm) ─────────────────
def std_of_others(group):
    tcr_vals = group["TCR"].values
    stds = [
        np.std(np.delete(tcr_vals, i), ddof=1) if len(tcr_vals) > 2 else np.nan
        for i in range(len(tcr_vals))
    ]
    group = group.copy()
    group["std_TCR"] = np.round(stds, 4)
    return group

merchants_df = merchants_df.groupby("year_month", group_keys=False).apply(std_of_others)

# ── Clean up and write ────────────────────────────────────────────────────────
merchants_df.drop(columns=["total_txns"], inplace=True)

merchants_df.to_sql("merchants", conn, if_exists="replace", index=False)
conn.close()
print(f"Done! {len(merchants_df)} rows, {len(merchants_df.columns)} columns written to merchants table.")
